
# Witness-Point Structural Training on the Helly Counterexample T_k

Axis-aligned boxes satisfy **Helly's theorem**: in $\mathbb{R}^d$, if
every pair among $k$ boxes has a non-empty intersection, then all
$k$ boxes share a common point. Consequently, any single-box embedding
(:class:`~mowl.models.ELBE`, :class:`~mowl.models.BoxSquaredEL`, or
:class:`~mowl.models.MultiBoxEL` with ``n_components=1``) cannot faithfully
model the Helly counterexample ontology family $\mathcal{T}_k$
($k \ge 3$), which defines $k$ classes

\begin{align}A_i \equiv \bigsqcap_{j \ne i} X_j \qquad (i = 1, \dots, k)\end{align}

from $k$ primitive classes $X_1, \dots, X_k$ and asserts that all
$\binom{k}{2}$ pairs are mutually disjoint:

\begin{align}A_i \sqcap A_j \sqsubseteq \bot \qquad (1 \le i < j \le k).\end{align}

In any model where every $A_i$ is non-empty, each ($k - 1$)-subset
of $\{X_1, \dots, X_k\}$ has a non-empty intersection, so single boxes
force $\bigcap_{j=1}^k X_j \ne \emptyset$ and therefore force every pair
$(A_i, A_j)$ to overlap.

Multi-component box embeddings (:class:`~mowl.models.MultiBoxEL` with
``n_components >= 2`` and ``objective="structural"``) have the capacity to
represent $\mathcal{T}_k$ in $d = 2$ dimensions with just two boxes
per primitive class $X_j$. However, naive structural training gets stuck
in a **component-assignment Helly trap**: gradient descent satisfies the
existence loss by routing all $k$ defined intersections $A_i$
through the *same* single component of each $X_j$, leaving the second
component idle and reproducing the single-box Helly obstruction across all
$\binom{k}{2}$ active slot pairs.

This tutorial builds $\mathcal{T}_k$ ($k \in \{3, 5\}$) with the
mOWL OWLAPI adapter, certifies single-box impossibility via
:func:`~mowl.geometric.helly.helly_check` and
:class:`~mowl.geometric.completion.CanonicalModel`, compares the round-4
default structural recipe against the witness-point recipe across three random
seeds, and visualises the resulting 2-D multi-box regions at $k = 5$.


## Imports and JVM initialisation
We cap PyTorch CPU threads to 1 so the small tensor operations per epoch run
without OpenMP thread-synchronisation overhead.



In [ ]:
import copy
import io
import logging
import os
import tempfile
import contextlib

import numpy as np
import torch as th

th.set_num_threads(1)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import mowl
mowl.init_jvm("2g")

import java.io
from org.semanticweb.owlapi.formats import FunctionalSyntaxDocumentFormat
from org.semanticweb.owlapi.model import IRI

from mowl.datasets import PathDataset
from mowl.geometric.completion import CanonicalModel
from mowl.geometric.helly import helly_check
from mowl.models import MultiBoxEL
from mowl.nn.el.structural import nonempty_measure, signed_overlap
from mowl.owlapi import OWLAPIAdapter

## Building $\mathcal{T}_k$ with the OWLAPI adapter
For $k \ge 3$, $\mathcal{T}_k$ has primitive classes
$X_1, \dots, X_k$ and defined classes
$A_i \equiv \sqcap_{j \ne i} X_j$ ($i = 1, \dots, k$). When
$k > 3$, each ($k - 1$)-way conjunction is written as a left-nested
chain of binary intersections using shared prefix definitions
``Aux_1_2``, ``Aux_1_2_3``, etc., so that binary normalisation produces a clean
binary definition DAG for :class:`~mowl.nn.el.structural.DefinitionTable`.
All ontology files and checkpoints are written under ``$TMPDIR`` (defaulting
to ``~/tmp``).



In [ ]:
NS = "http://mowl.borg/geo#"
TMP_ROOT = os.environ.get("TMPDIR") or os.path.expanduser("~/tmp")
os.makedirs(TMP_ROOT, exist_ok=True)
WORK_DIR = tempfile.mkdtemp(prefix="mowl_tk_tutorial_", dir=TMP_ROOT)


def build_tk_dataset(k: int, out_dir: str) -> PathDataset:
    """Build the Helly counterexample ontology T_k and wrap it in a PathDataset."""
    adapter = OWLAPIAdapter()
    ont = adapter.create_ontology(f"http://mowl.borg/geo/t{k}")
    mgr = adapter.owl_manager

    xs = [adapter.create_class(f"{NS}X{j + 1}") for j in range(k)]
    as_ = [adapter.create_class(f"{NS}A{i + 1}") for i in range(k)]

    prefix_cache = {}

    def get_prefix(indices):
        if len(indices) == 1:
            return xs[indices[0]]
        if indices in prefix_cache:
            return prefix_cache[indices]
        left = get_prefix(indices[:-1])
        right = xs[indices[-1]]
        label = "_".join(str(idx + 1) for idx in indices)
        aux = adapter.create_class(f"{NS}Aux_{label}")
        conj = adapter.create_object_intersection_of(left, right)
        mgr.addAxiom(ont, adapter.create_equivalent_classes(aux, conj))
        prefix_cache[indices] = aux
        return aux

    for i in range(k):
        others = tuple(j for j in range(k) if j != i)
        rhs = adapter.create_object_intersection_of(
            get_prefix(others[:-1]), xs[others[-1]]
        )
        mgr.addAxiom(ont, adapter.create_equivalent_classes(as_[i], rhs))

    bot = adapter.data_factory.getOWLNothing()
    for i in range(k):
        for j in range(i + 1, k):
            disj = adapter.create_object_intersection_of(as_[i], as_[j])
            mgr.addAxiom(ont, adapter.create_subclass_of(disj, bot))

    owl_path = os.path.join(out_dir, f"t{k}.owl")
    doc_iri = IRI.create(java.io.File(str(owl_path)).toURI())
    mgr.saveOntology(ont, FunctionalSyntaxDocumentFormat(), doc_iri)
    return PathDataset(owl_path)

## Why single boxes cannot model $\mathcal{T}_k$: Helly check & canonical model
Before training any neural embedding, we inspect the canonical model
:class:`~mowl.geometric.completion.CanonicalModel` and run
:func:`~mowl.geometric.helly.helly_check` with ``k_max=k``.

In $\mathcal{T}_k$, each defined class $A_i$ is a satisfiable
canonical witness whose closure contains every $X_j$ with $j \ne i$,
while the $k$-tuple $\{X_1, \dots, X_k\}$ is jointly unsatisfiable.
By Helly's theorem for axis-aligned boxes in $\mathbb{R}^d$, any family
of single boxes whose pairwise intersections are non-empty must have a common
point across the entire family, so :func:`~mowl.geometric.helly.helly_check`
returns ``box_representable_strong=False``.



In [ ]:
datasets = {}
canonical_models = {}

for k in (3, 5):
    ds = build_tk_dataset(k, WORK_DIR)
    cm = CanonicalModel.from_dataset(ds)
    report = helly_check(cm, k_max=k)
    datasets[k] = ds
    canonical_models[k] = cm

    a_witnesses = [
        f"A{i + 1}" for i in range(k) if cm.is_satisfiable(f"{NS}A{i + 1}")
    ]
    print(
        f"T_{k}: satisfiable A-witnesses={a_witnesses}, "
        f"box_representable_strong={report.box_representable_strong}, "
        f"strong_certificates={len(report.certificates_strong)}"
    )

## Structural losses, the Helly trap, and the three witness-point ingredients
Under ``objective="structural"``, :class:`~mowl.models.MultiBoxEL` compiles
binary equivalences $M \equiv A \sqcap B$ into a topological
:class:`~mowl.nn.el.structural.DefinitionTable` and evaluates defined classes
constructively by intersecting the component boxes of their operands
(:meth:`~mowl.nn.MultiBoxELModule.all_class_regions`). Training optimises:

* **Existence** (:func:`~mowl.nn.el.structural.existence_loss` via
  :func:`~mowl.nn.el.structural.nonempty_measure`): every satisfiable class
  must have at least one active slot whose minimum half-width exceeds
  ``margin``.
* **Disjointness** (:func:`~mowl.nn.el.structural.gci1_bot_structural` via
  :func:`~mowl.nn.el.structural.signed_overlap`): for every
  $A_i \sqcap A_j \sqsubseteq \bot$, every pair of non-empty active
  slots must be separated by at least ``margin`` along some coordinate.

**Why the default structural recipe gets trapped.** Even with
``n_components=2`` boxes per primitive class $X_j$, the round-4 defaults
(``bot_agg="mean"``, ``witness_weight=0.0``, ``reseed_every=0``,
``reseed_mode="dead"``, ``size_reg=1e-3``) allow every $A_i$ to satisfy
existence using the *same* single component of each $X_j$. Restricted to
one active box per $X_j$, Helly's theorem applies again: whenever all
$k$ classes $A_1, \dots, A_k$ are non-empty, their single active
slots share a common intersection and all $\binom{k}{2}$ pairs
$(A_i, A_j)$ overlap.

**The three round-5 ingredients** break this symmetry:

1. **Sum aggregation** (``bot_agg="sum"``): sums the squared-hinge overlap
   penalties over all active slot pairs of $(C, D)$ in
   :func:`~mowl.nn.el.structural.gci1_bot_structural` instead of averaging
   them (``bot_agg="mean"``), preventing the $1 / (S_C S_D)$ dilution as
   the number of definition slots grows.
2. **Canonical-model witness points** (``witness_weight=1.0``): compiles a
   :class:`~mowl.nn.el.structural.WitnessTargets` bundle from the canonical
   model and attaches one learnable point $p(w) \in \mathbb{R}^d$ per
   satisfiable class (:meth:`~mowl.nn.MultiBoxELModule.witness_points`). Using
   :func:`~mowl.nn.el.structural.inside_measure`,
   :func:`~mowl.nn.el.structural.witness_loss` pulls $p(A_i)$ inside at
   least one active component of every $X_j$ ($j \ne i$) and pushes
   it outside *every* active component of $X_i$ (since
   $A_i \sqcap X_i \sqsubseteq \bot$ is entailed by the canonical model).
3. **Idle-component re-seeding** (``reseed_mode="idle"``, ``reseed_every=50``,
   ``reseed_size=0.15``, ``reseed_batch=1``, with ``size_reg=0.0``):
   periodically traces which primitive components participate in live
   definition slots via :func:`~mowl.nn.el.structural.slot_components` and
   relocates the *least-used* (idle) component of each missing conjunct onto
   the dead definition's witness point $p(A_i)$ without disturbing the
   component already serving live definitions.



In [ ]:
def evaluate_tk(model: MultiBoxEL, k: int):
    """Return ``(is_exact_model, n_nonempty_a, overlapping_slot_pairs)``."""
    with th.no_grad():
        regions = model.module.all_class_regions()
        targets = model.module.existence_targets

        sat_ok = True
        for cls_idx in targets.classes.tolist():
            c = int(cls_idx)
            ne_c = nonempty_measure(regions.w[c])[regions.active[c]]
            if float(ne_c.max().item()) <= 0.0:
                sat_ok = False

        cid = model.class_index_dict
        a_indices = [cid[f"{NS}A{i + 1}"] for i in range(k)]
        n_nonempty_a = sum(
            int(
                float(nonempty_measure(regions.w[ci])[regions.active[ci]].max().item())
                > 0.0
            )
            for ci in a_indices
        )

        ov_pairs = 0
        for i in range(k):
            for j in range(i + 1, k):
                ci, cj = a_indices[i], a_indices[j]
                sov = signed_overlap(
                    regions.cen[ci : ci + 1],
                    regions.w[ci : ci + 1],
                    regions.cen[cj : cj + 1],
                    regions.w[cj : cj + 1],
                )
                pair_ne = th.amin(sov, dim=-1)[0]
                act_pairs = regions.active[ci].unsqueeze(1) & regions.active[
                    cj
                ].unsqueeze(0)
                ov_pairs += int((pair_ne[act_pairs] > 0.0).sum().item())

        gci1_bot_rows = model.training_datasets["gci1_bot"][:][:, :2].detach().cpu()
        bot_ok = True
        for row in gci1_bot_rows.tolist():
            ci, cj = int(row[0]), int(row[1])
            sov = signed_overlap(
                regions.cen[ci : ci + 1],
                regions.w[ci : ci + 1],
                regions.cen[cj : cj + 1],
                regions.w[cj : cj + 1],
            )
            pair_ne = th.amin(sov, dim=-1)[0]
            act_pairs = regions.active[ci].unsqueeze(1) & regions.active[
                cj
            ].unsqueeze(0)
            if float(pair_ne[act_pairs].max().item()) > 0.0:
                bot_ok = False

        return bool(sat_ok and bot_ok), int(n_nonempty_a), int(ov_pairs)

## Training experiment: 2 recipes $\times$ $k \in \{3, 5\}$ $\times$ 3 seeds
We train :class:`~mowl.models.MultiBoxEL` in $d = 2$ dimensions with
``n_components=2`` and ``max_components=2**(k - 1)`` (so every $A_i$ has
$2^{k-1}$ slots). During training, a lightweight callback records the
best geometric state—prioritising an exact model first, and otherwise the
epoch where the largest number of $A_i$ witnesses are simultaneously
non-empty (exposing the $\binom{k}{2}$ overlapping slot pairs of the
Helly trap).



In [ ]:
RECIPES = {
    "default": dict(
        bot_agg="mean",
        witness_weight=0.0,
        reseed_every=0,
        reseed_mode="dead",
        size_reg=1e-3,
    ),
    "witness": dict(
        bot_agg="sum",
        witness_weight=1.0,
        reseed_every=50,
        reseed_mode="idle",
        reseed_size=0.15,
        reseed_batch=1,
        size_reg=0.0,
    ),
}

SEEDS = (0, 1, 2)
EPOCHS_BY_K = {3: 600, 5: 1200}

summary_rows = []
saved_k5_models = {}

for k in (3, 5):
    ds = datasets[k]
    cm = canonical_models[k]
    max_comp = 2 ** (k - 1)
    epochs = EPOCHS_BY_K[k]

    for recipe_name, recipe_kw in RECIPES.items():
        exact_count = 0
        ov_counts = []

        for seed in SEEDS:
            th.manual_seed(seed)
            np.random.seed(seed)
            ckpt_path = os.path.join(WORK_DIR, f"mb_t{k}_{recipe_name}_s{seed}.pt")

            best = {
                "key": (-1, -1, -999),
                "metrics": (False, 0, 0),
                "state": None,
            }

            def callback(epoch, m, _best=best, _k=k):
                if (epoch + 1) % 20 != 0 and epoch != epochs - 1:
                    return
                is_exact, n_ne_a, ov_pairs = evaluate_tk(m, _k)
                key = (int(is_exact), n_ne_a, -ov_pairs)
                if key > _best["key"]:
                    _best["key"] = key
                    _best["metrics"] = (is_exact, n_ne_a, ov_pairs)
                    _best["state"] = copy.deepcopy(m.module.state_dict())
                if is_exact:
                    m.reseed_every = 0

            logging.disable(logging.WARNING)
            try:
                with contextlib.redirect_stdout(
                    io.StringIO()
                ), contextlib.redirect_stderr(io.StringIO()):
                    model = MultiBoxEL(
                        ds,
                        embed_dim=2,
                        n_components=2,
                        max_components=max_comp,
                        margin=0.02,
                        learning_rate=0.01,
                        objective="structural",
                        points_seed=seed,
                        canonical_model=cm,
                        neg_sampling_gcis=[],
                        model_filepath=ckpt_path,
                        **recipe_kw,
                    )
                    model.train(
                        epochs=epochs,
                        validate_every=epochs + 1,
                        epoch_callback=callback,
                    )
            finally:
                logging.disable(logging.NOTSET)

            if best["state"] is not None:
                model.module.load_state_dict(best["state"])

            is_exact, n_ne_a, ov_pairs = evaluate_tk(model, k)
            exact_count += int(is_exact)
            ov_counts.append(ov_pairs)

            if k == 5 and (
                recipe_name not in saved_k5_models
                or (is_exact and not saved_k5_models[recipe_name][1][0])
            ):
                saved_k5_models[recipe_name] = (model, (is_exact, n_ne_a, ov_pairs), seed)

        summary_rows.append((k, recipe_name, exact_count, ov_counts))

print(f"{'k':>3} | {'recipe':<10} | {'exact models /3':>15} | {'overlapping slot pairs':<24}")
print("-" * 62)
for k, recipe_name, exact_count, ov_counts in summary_rows:
    print(
        f"{k:>3} | {recipe_name:<10} | {f'{exact_count}/3':>15} | {str(ov_counts):<24}"
    )

## Visualising the 2-D regions at $k = 5$
For $k = 5$, we plot the learned 2-D primitive class components
$X_1, \dots, X_5$ for both recipes:

* **Left (default recipe)**: all $A_i$ share a single component per
  primitive class $X_j$, so the 5 active boxes overlap in a common
  central region and all $\binom{5}{2} = 10$ slot pairs among
  $A_1, \dots, A_5$ intersect.
* **Right (witness-point recipe)**: each primitive class $X_j$ splits
  its two components (solid and dashed rectangles of the same colour) around
  the excluded witness point $p(A_j)$, creating 5 disjoint intersection
  pockets—one around each witness marker $p(A_1), \dots, p(A_5)$—with
  ``0`` overlapping slot pairs.



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.8))
palette = plt.cm.tab10(np.linspace(0.0, 0.5, 5))

panel_specs = [
    ("default", "Round-4 default recipe (Helly trap)"),
    ("witness", "Round-5 witness-point recipe (Exact model)"),
]

for ax, (recipe_name, title_prefix) in zip(axes, panel_specs):
    model, (is_exact, n_ne_a, ov_pairs), seed = saved_k5_models[recipe_name]
    with th.no_grad():
        regions = model.module.all_class_regions()
        pts = model.module.witness_points()

    cid = model.class_index_dict
    xs_coords, ys_coords = [], []

    for j in range(5):
        cls_iri = f"{NS}X{j + 1}"
        cls_idx = cid[cls_iri]
        color = palette[j]

        for c in range(model.n_components):
            if not bool(regions.active[cls_idx, c].item()):
                continue
            w_c = regions.w[cls_idx, c]
            if float(nonempty_measure(w_c).item()) <= 0.0:
                continue
            cx, cy = (
                float(regions.cen[cls_idx, c, 0].item()),
                float(regions.cen[cls_idx, c, 1].item()),
            )
            hx, hy = float(w_c[0].item()), float(w_c[1].item())
            xs_coords.extend([cx - hx, cx + hx])
            ys_coords.extend([cy - hy, cy + hy])

            rect = mpatches.Rectangle(
                (cx - hx, cy - hy),
                2.0 * hx,
                2.0 * hy,
                facecolor=color,
                alpha=0.10,
                edgecolor=color,
                linestyle="solid" if c == 0 else "dashed",
                linewidth=1.8,
                label=f"X{j + 1}" if c == 0 else None,
            )
            ax.add_patch(rect)

    if (
        pts is not None
        and model.module.existence_targets is not None
        and model.module.existence_targets.witness is not None
    ):
        w_classes = model.module.existence_targets.witness.classes.tolist()
        for i in range(5):
            a_idx = cid[f"{NS}A{i + 1}"]
            if a_idx in w_classes:
                w_pos = w_classes.index(a_idx)
                px = float(pts[w_pos, 0].item())
                py = float(pts[w_pos, 1].item())
                xs_coords.append(px)
                ys_coords.append(py)
                ax.scatter(
                    [px],
                    [py],
                    color=palette[i],
                    edgecolor="black",
                    marker="*",
                    s=180,
                    zorder=6,
                    linewidth=0.9,
                    label="witness $p(A_i)$" if i == 0 else None,
                )
                ax.annotate(
                    f"p(A{i + 1})",
                    (px, py),
                    textcoords="offset points",
                    xytext=(6, 6),
                    fontsize=9,
                    fontweight="bold",
                    color="black",
                    zorder=7,
                )

    pad = 0.2
    if xs_coords and ys_coords:
        ax.set_xlim(min(xs_coords) - pad, max(xs_coords) + pad)
        ax.set_ylim(min(ys_coords) - pad, max(ys_coords) + pad)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, alpha=0.25)
    ax.set_xlabel("Dimension 1")
    ax.set_ylabel("Dimension 2")
    ax.set_title(
        f"{title_prefix}\n"
        f"(seed={seed}, exact={is_exact}, non-empty A_i={n_ne_a}/5, "
        f"overlapping slot pairs={ov_pairs})",
        fontsize=10.5,
    )
    ax.legend(loc="upper right", fontsize=8.5, framealpha=0.9)

fig.suptitle(
    "T_5 primitive class regions X_1..X_5 (solid = comp 0, dashed = comp 1) and witness points",
    fontsize=12,
)
plt.tight_layout()
plt.show()